# keyboard

> Self-contained keyboard system for sortable queue navigation and actions.

In [ ]:
#| default_exp keyboard

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from typing import Optional

from cjm_fasthtml_keyboard_navigation.core.focus_zone import FocusZone
from cjm_fasthtml_keyboard_navigation.core.navigation import LinearVertical
from cjm_fasthtml_keyboard_navigation.core.actions import KeyAction
from cjm_fasthtml_keyboard_navigation.core.manager import ZoneManager
from cjm_fasthtml_keyboard_navigation.components.system import render_keyboard_system, KeyboardSystem

from cjm_fasthtml_sortable_queue.config import SortableQueueConfig
from cjm_fasthtml_sortable_queue.html_ids import SortableQueueHtmlIds
from cjm_fasthtml_sortable_queue.models import SortableQueueUrls

In [ ]:
#| export
def create_queue_keyboard_system(
    config: SortableQueueConfig,  # Queue configuration
    ids: SortableQueueHtmlIds,  # HTML ID generators
    urls: SortableQueueUrls,  # URL endpoints for HTMX actions
    zone_focus_classes: tuple = (),  # CSS classes when queue zone is active
    item_focus_classes: tuple = (),  # CSS classes on focused item
    data_attributes: tuple = (),  # Data attributes to extract (e.g., ("record-id", "provider-id"))
    on_focus_change: Optional[str] = None,  # JS callback on item focus change
    hidden_input_prefix: Optional[str] = None,  # Prefix for hidden state inputs
    system_id: Optional[str] = None,  # Keyboard system ID (auto-generated from ids.system_id if not set)
    show_hints: bool = False,  # Show keyboard hints UI
) -> KeyboardSystem:  # Complete rendered keyboard system
    """Create a self-contained keyboard system for the sortable queue.

    Returns a rendered KeyboardSystem with a single FocusZone (LinearVertical
    navigation) and built-in actions for Delete/Backspace remove and
    Shift+Arrow reorder. Works standalone or as a child in a hierarchy
    via `coord.setParent(system_id, parent_id)`.
    """
    # Build the queue focus zone
    zone_kwargs = dict(
        id=ids.container,
        item_selector=f"li.{config.item_class}",
        navigation=LinearVertical(),
    )
    if zone_focus_classes:
        zone_kwargs["zone_focus_classes"] = zone_focus_classes
    if item_focus_classes:
        zone_kwargs["item_focus_classes"] = item_focus_classes
    if data_attributes:
        zone_kwargs["data_attributes"] = data_attributes
    if on_focus_change:
        zone_kwargs["on_focus_change"] = on_focus_change
    if hidden_input_prefix:
        zone_kwargs["hidden_input_prefix"] = hidden_input_prefix
    
    queue_zone = FocusZone(**zone_kwargs)
    
    # Built-in queue actions
    queue_zone_ids = (ids.container,)
    actions = (
        # Remove focused item (Delete)
        KeyAction(
            key="Delete",
            htmx_trigger=ids.remove_btn,
            zone_ids=queue_zone_ids,
            description="Remove from queue",
            hint_group="Queue",
        ),
        # Remove focused item (Backspace) — hidden alias
        KeyAction(
            key="Backspace",
            htmx_trigger=ids.remove_btn,
            zone_ids=queue_zone_ids,
            description="Remove from queue",
            hint_group="Queue",
            show_in_hints=False,
        ),
        # Reorder up (Shift+ArrowUp)
        KeyAction(
            key="ArrowUp",
            modifiers=frozenset({"shift"}),
            htmx_trigger=ids.reorder_up_btn,
            zone_ids=queue_zone_ids,
            description="Move up in queue",
            hint_group="Queue",
        ),
        # Reorder down (Shift+ArrowDown)
        KeyAction(
            key="ArrowDown",
            modifiers=frozenset({"shift"}),
            htmx_trigger=ids.reorder_down_btn,
            zone_ids=queue_zone_ids,
            description="Move down in queue",
            hint_group="Queue",
        ),
    )
    
    # Assemble ZoneManager
    manager = ZoneManager(
        zones=(queue_zone,),
        actions=actions,
        system_id=system_id or ids.system_id,
        state_hidden_inputs=True,
    )
    
    # URL/target/swap/vals maps for the hidden action buttons
    container_selector = ids.as_selector(ids.container)
    url_map = {
        ids.remove_btn: urls.remove,
        ids.reorder_up_btn: urls.reorder,
        ids.reorder_down_btn: urls.reorder,
    }
    target_map = {
        ids.remove_btn: container_selector,
        ids.reorder_up_btn: container_selector,
        ids.reorder_down_btn: container_selector,
    }
    swap_map = {
        ids.remove_btn: "outerHTML",
        ids.reorder_up_btn: "outerHTML",
        ids.reorder_down_btn: "outerHTML",
    }
    # Direction values distinguish up/down reorder buttons hitting the same URL
    vals_map = {
        ids.reorder_up_btn: {"direction": "up"},
        ids.reorder_down_btn: {"direction": "down"},
    }
    
    return render_keyboard_system(
        manager=manager,
        url_map=url_map,
        target_map=target_map,
        swap_map=swap_map,
        vals_map=vals_map,
        show_hints=show_hints,
    )

## Tests

In [ ]:
from fasthtml.common import to_xml

# Test setup
config = SortableQueueConfig(prefix="tk")
ids = SortableQueueHtmlIds(prefix="tk")
urls = SortableQueueUrls(reorder="/tk/reorder", remove="/tk/remove", clear="/tk/clear")

# --- Basic creation ---
kb = create_queue_keyboard_system(config, ids, urls)
assert isinstance(kb, KeyboardSystem)
assert kb.script is not None
assert kb.hidden_inputs is not None
assert kb.action_buttons is not None

# --- System ID ---
# Default system_id comes from ids.system_id
script_xml = to_xml(kb.script)
assert "tk-queue-kb" in script_xml  # system_id appears in JS config

# Custom system_id
kb2 = create_queue_keyboard_system(config, ids, urls, system_id="my-custom-kb")
script_xml2 = to_xml(kb2.script)
assert "my-custom-kb" in script_xml2

# --- Action buttons ---
btns_xml = to_xml(kb.action_buttons)
assert ids.remove_btn in btns_xml  # Remove button present
assert ids.reorder_up_btn in btns_xml  # Reorder up button present
assert ids.reorder_down_btn in btns_xml  # Reorder down button present
assert "/tk/remove" in btns_xml  # Remove URL wired
assert "/tk/reorder" in btns_xml  # Reorder URL wired

# --- Direction vals on reorder buttons ---
assert '"direction": "up"' in btns_xml or "'direction': 'up'" in btns_xml or "direction" in btns_xml
assert '"direction": "down"' in btns_xml or "'direction': 'down'" in btns_xml

# --- Zone config in script ---
assert f"li.{config.item_class}" in script_xml  # Item selector
assert ids.container in script_xml  # Zone container ID

# --- Optional parameters ---
kb3 = create_queue_keyboard_system(
    config, ids, urls,
    zone_focus_classes=("ring-1", "ring-secondary"),
    item_focus_classes=("bg-secondary/10",),
    data_attributes=("record-id", "provider-id"),
    on_focus_change="myFocusCallback",
    hidden_input_prefix="my-focused",
    show_hints=True,
)
assert isinstance(kb3, KeyboardSystem)
assert kb3.hints is not None  # Hints rendered when show_hints=True
script_xml3 = to_xml(kb3.script)
assert "myFocusCallback" in script_xml3
assert "record-id" in script_xml3

print("All keyboard tests passed")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()